# Note

This notebook is intended as an exploration of the available data.
Much of the code for creating summary plots has been removed for you to complete.
Feel free to also add additional cells and explore the data yourself and together with your peers.

If you get stuck or would like to see our solution, take a look in `2-SOLUTION.ipynb`

The other notebooks in the folder are complete. Once you finish this notebook, you can work through them and experiment. Have fun!

# Imports

Import the needed packages here.
This already imports the course library - you can add your own functions there if you want them to be usable in every notebook or script for this course.

In [ ]:
%load_ext autoreload
%autoreload 2

import megcourselib
import mne
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns

import os
from os import path

# Set paths and constants

If you followed the instructions for downloading (or linking) the data for this course, you should not need to change this path.
`PROJECT_ROOT` points to the root of the course repository. The path can be absolute (e.g. /Users/myuser/Documents/MEG_encoding_course), or relative to the working directory of your jupyter server. The reason we set this as a variable is because different editors can handle this working directory differently.
If you opened this notebook in your browser, the working directory is likely the notebooks folder. In VSCode, it might be the root of the project you have opened.
To find out, we can use the `!` syntax to execute a command in your shell:

In [ ]:
!pwd

In [ ]:
PROJECT_ROOT = ".."
CHALLENGE = "challenge1"
DATA_ROOT = path.join(PROJECT_ROOT, 'data', 'brainencoding26', CHALLENGE)
STIM_ROOT = path.join(PROJECT_ROOT, 'data', 'avs_scenes')

In [ ]:
# list files
files = os.listdir(DATA_ROOT)
print(f"dataset folder at {DATA_ROOT} contains the following files:")
for f in files: print(f)

We can also pass python variables to the shell from jupyter. Note the `$` in front of the variable name.  
Here, we show the organisation of the dataset (This may not work on Windows because the `tree` command may not be available, alternatively you can look at the file tree in your favorite editor):

In [ ]:
!tree $DATA_ROOT

# Load Metadata

In [ ]:
metadata = pd.read_csv(path.join(DATA_ROOT, 'training', 'metadata.csv'))

print('Found keys:')
print(metadata.keys())

In [ ]:
# make sure pandas does not hide columns
pd.set_option('display.max_columns', None)
# show a few rows of the dataframe
metadata

## Get an overview of the data

Explore the pandas dataframe that has been loaded and try to answer some of the questions below.
Discuss with your peers if you get stuck or want to compare your results.
A good way to get a feeling for the dataset is to create some plots. 
(For pandas data frames, seaborn is an excellent tool for this, see the example below).

1. How many participants are there?
2. How many sessions has each participant completed?
3. How many trials are there?
4. How many fixations are there?
5. What is the distribution of fixation lengths for each participant?
6. How many fixations are there on average per scene. What does the distribution look like? Is it similar for all participants?
7. What other interesting information can you extract from the metadata?

### Fixation lengths

Tip: Many seaborn plots allow keyword arguments 'hue', 'row', 'col' (among others, have a look at the documentation for more information).
This makes it easy to split and visualize your data in different ways.

Try changing the plot below to show the fixation distribution in different colors for each participant. Alternatively, show a row of panels for each participant, and color the distributions by session

In [ ]:
sns.displot(
    data=metadata,
    x='duration',
    # palette='tab10'
)
plt.title('Distribution of fixation durations')

### Trial and fixation counts

In [ ]:
sns.set_theme(
    context='notebook', style='ticks', palette='deep', font='sans-serif', font_scale=1
)

# get unique subjects
subjects = metadata['subject'].unique()

# for each subject, count the number of unique trials, and fixations

# -------------- your code here 


### Fixations per scene

In [ ]:
# we need the subject and scene columns for grouping

# -------------- your code here 


### Feel free to explore below!

In [ ]:
# --- your plots here



# Load Stimuli

In this section we inspect the stimuli that were shown to participants.
Scenes are stored with standardized filenames. From the sceneID, we can get the path to an image and load it.
As a first step, we ensure all the stimuli can be loaded:

In [ ]:
"""
First, we locate the stimuli.
"""

def get_scene_path(scene_id, stim_root, id_str_len=12, strict=True):
    # make sure scene_id is an integer
    scene_id = int(scene_id)
    # now convert to str
    scene_str = str(scene_id)
    # pad with zeros to match filename
    scene_str = scene_str.zfill(id_str_len)
    # assemble filename
    fname = f"{scene_str}_MEG_size.jpg"
    # assemble path
    fpath = path.join(stim_root, fname)
    if strict:
        assert path.exists(fpath), f"{fpath} does not exist."
    return fpath

# check for missing data.
for scene_id in metadata.sceneID:
    get_scene_path(scene_id, STIM_ROOT)
print('Stimulus set complete.')

Next, we can look at a few stimuli. Change `IDX` to look at the scene shown for different fixations in the metadata dataframe.

In [ ]:
IDX = 42

scene_id = metadata.sceneID[IDX]
image_path = get_scene_path(scene_id, STIM_ROOT)
im = Image.open(image_path).convert('RGB')
im_arr = np.asarray(im)

plt.figure(figsize=(8,6))
plt.imshow(im_arr)
plt.axis('off')
plt.title(f"Scene ID: {int(scene_id)}")
plt.show()


# Fixations

For each fixation, we know the exact location that was fixated by the participant. We can use this to get crops from the image to generate inputs for our encoding models.

In [ ]:
from megcourselib.data.stim import extract_crop

gx = metadata.iloc[IDX].mean_gx
gy = metadata.iloc[IDX].mean_gy
scene_id = metadata.sceneID[IDX]
fix_number = int(metadata.fix_sequence[IDX])

print('fixation location:', gx, gy)
print('sceneID:', scene_id)
print('fixation number:', fix_number)

# use the extract_crop function to show the crop for a single fixation

# -------------- your code here 


## Plot all fixations for a scene

Exercise: 

For a given subject, and scene id. extract crops for all fixations and show them in order.
Note, for fixations close to the edge of an image, large crops may exceed the limits of the image. By default `extract_crop` will return a black image.

In [ ]:
subject = 2
scene_id = 578493


df = metadata[
    # filter the metadata to get only rows for the selected subject and scene_id
]

# make sure fixations are sorted in order
df = df.sort_values(
    by='fix_sequence'
)

# -------------- create a plot with all fixations, similar to the previous cell


## Plot fixations as a scatter plot

There already is a function to show a scene with fixation centers as a scatter plot. It is imported below - find it in the repository and try to understand what it does.


Then, create this plot for the current scene:

In [ ]:
from megcourselib.plotting.stim import plot_scene_fixations

subject = 2
scene_id = 578493

df = metadata[
    # filter the metadata as you did before
]

# get image (also as before)
image_path = get_scene_path(scene_id, STIM_ROOT)
im = Image.open(image_path).convert('RGB')

fig = plot_scene_fixations(scene_id, im, df)

# Load MEG dataset

Finally, load the MEG data as well. For the first challenge, we have measurements from all sensors for a single time point 110ms after fixation onset. Below is an example for how to load and plot this data.

In [ ]:
meg_train = np.load(path.join(DATA_ROOT, 'training', 'meg_110ms.npy'))
meg_train.shape

In [ ]:
subject = 2
scene_id = 578493
fixation = 0

# get information about sensor location for the subject
fif_path = path.join(DATA_ROOT, 'training', f"sub-{str(subject).zfill(2)}_grad_info.fif")
info = mne.io.read_info(fif_path)

# get metadata
fixmeta = metadata[
    (metadata.subject == subject) &
    (metadata.sceneID == scene_id) &
    (metadata.fix_sequence == fixation)
]
assert len(fixmeta) == 1, 'constraint matches more than one fixation'
idx = fixmeta.index[0]
fixmeta = fixmeta.iloc[0]

# plot data
fig, ax = plt.subplots(figsize=(6, 6))
im, _ = mne.viz.plot_topomap(
    data=meg_train[idx],
    pos=info,
    ch_type='grad',
    axes=ax,
    show=False          # prevent auto-display so we can add the colorbar
)

plt.colorbar(im, ax=ax, shrink=0.7, label='Field strength (T/m)')
plt.title(f"MEG measurement for subj {subject} - scene id {scene_id} - fixation {fixation}")
plt.tight_layout()
plt.show()

# Next Steps

That's it. You should now have an overview of the data available for the first challenge. Feel free to explore some more to get an intuition for the MEG data (e.g. what is the range of measured values for each participant and sensor? What is the distribution like? Are there many outliers?)

Once you are content, you can move on to the next notebook for an introduction to feature extraction to build towards your first encoding model.

In [ ]:
# --- your exploration here 